# 01 · Limpieza y frecuencias

**Duración estimada:** 35 minutos

## Propósito

Contar palabras. Es lo más simple que se puede hacer con un corpus, y es donde
están las trampas más grandes.

Vamos a contar mal primero, ver por qué está mal, y arreglarlo.

## Qué consume

- `datos/corpus.jsonl`
- `datos/stopwords_es.txt`

## Celda de reanudación

Si llegaste tarde o se reinició el kernel, corré esta celda y seguís desde acá.

In [ ]:
# pandas: para trabajar el corpus como tabla.
import pandas as pd

# Counter: para contar cosas. Devuelve un diccionario palabra -> veces.
from collections import Counter

corpus = pd.read_json("datos/corpus.jsonl", lines=True)

print("párrafos:", f"{len(corpus):,}")
print("años    :", sorted(corpus["anio"].unique()))

## 1. Contar como saldría naturalmente

Juntemos todo el corpus en un solo texto, cortemos por espacios y contemos.

In [ ]:
todo_el_corpus = " ".join(corpus["texto"])

palabras = todo_el_corpus.split()

conteo = Counter(palabras)

print("palabras en total  :", f"{len(palabras):,}")
print("palabras distintas :", f"{len(conteo):,}")
print()
print("las 15 más frecuentes:")
for palabra, veces in conteo.most_common(15):
    print(" ", palabra, "->", veces)

Dos problemas, y los dos se ven a simple vista.

**Primero:** las quince palabras más frecuentes son artículos, preposiciones y
conjunciones. No dicen nada sobre de qué hablan los textos.

**Segundo, y peor:** buscá `de` y `De` en esa lista, o `que` y `Que`. Para el
contador son palabras distintas. Y hay algo más.

In [ ]:
filas = []
for palabra in ["que", "Que", "casa", "casa.", "casa,", "Casa"]:
    fila = {"entrada_del_vocabulario": palabra, "veces": conteo[palabra]}
    filas.append(fila)

pd.DataFrame(filas)

`casa`, `casa.` y `casa,` son tres entradas distintas del vocabulario. La misma
palabra, contada tres veces por separado, según qué signo tenía al lado.

Eso es lo que arregla la limpieza.

## 2. Limpiar

Tres operaciones, una por vez, para ver qué hace cada una.

In [ ]:
# re: expresiones regulares.
import re

# string: trae la lista de signos de puntuación.
import string

ejemplo = "¡Hola! ¿Cómo estás? En 2001 gané la F1 —dijo—, y me fui a casa."

print("original          :", ejemplo)

# Operación 1: pasar a minúsculas.
paso_1 = ejemplo.lower()
print("en minúsculas     :", paso_1)

In [ ]:
# Operación 2: sacar la puntuación.
# string.punctuation trae los signos del inglés.
print("signos que trae string.punctuation:")
print(string.punctuation)

Fijate qué **no** está ahí: `¡`, `¿`, las comillas tipográficas `“ ”`, la raya
de diálogo `—` y los puntos suspensivos `…`.

Son todos signos que aparecen en un texto en español. Si usamos la lista tal
cual, quedan adentro del vocabulario.

In [ ]:
signos_del_espanol = "¡¿—–…“”‘’«»"
todos_los_signos = string.punctuation + signos_del_espanol

# re.escape prepara los signos para que la expresión regular los tome literalmente.
patron_de_signos = "[" + re.escape(todos_los_signos) + "]"

paso_2 = re.sub(patron_de_signos, " ", paso_1)
print("sin puntuación    :", paso_2)

In [ ]:
# Operación 3: unificar los espacios múltiples que quedaron.
paso_3 = re.sub(r"\s+", " ", paso_2).strip()

print("resultado         :", paso_3)

### La decisión que no vamos a tomar

En el material del que salió este ejercicio había una cuarta operación:

```python
texto = re.sub(r'\w*\d\w*', ' ', texto)   # borrar palabras con números
```

Parece razonable: saca códigos y basura numérica. Veamos qué hace acá.

In [ ]:
print("con esa línea:", re.sub(r"\w*\d\w*", " ", paso_3))

In [ ]:
# ¿Cuántos años de cuatro dígitos hay en el corpus?
anios_mencionados = re.findall(r"\b(?:19|20)\d{2}\b", todo_el_corpus)

print("menciones a años en el corpus:", f"{len(anios_mencionados):,}")
print()
print("los más mencionados:")
conteo_anios = Counter(anios_mencionados)
for anio, veces in conteo_anios.most_common(8):
    print(" ", anio, "->", veces)

**Esa línea borra 447 menciones a años** en un corpus cuya única variable es el
tiempo.

*"En 2001 gané la F1"* queda *"en gané la"*.

No la vamos a usar. Y el punto no es que esté mal en general —en otro corpus
puede estar perfecta— sino que **una operación de limpieza siempre destruye algo,
y hay que saber qué.**

> **Limpiar no es corregir errores.** Es decidir qué diferencias nos importan y
> cuáles no. Cada línea de limpieza es una afirmación sobre el corpus.

### La función completa

In [ ]:
def limpiar(texto):
    # Pasa a minúsculas, saca la puntuación y unifica los espacios.
    # NO saca números: en este corpus los años son datos.
    en_minusculas = texto.lower()
    sin_signos = re.sub(patron_de_signos, " ", en_minusculas)
    sin_espacios_de_mas = re.sub(r"\s+", " ", sin_signos)
    return sin_espacios_de_mas.strip()


# La aplicamos a todo el corpus.
textos_limpios = []
for texto in corpus["texto"]:
    textos_limpios.append(limpiar(texto))

corpus["texto_limpio"] = textos_limpios

corpus.head(3)

## 3. Contar de nuevo

In [ ]:
todo_limpio = " ".join(corpus["texto_limpio"])
palabras_limpias = todo_limpio.split()
conteo_limpio = Counter(palabras_limpias)

print("antes  ->", f"{len(conteo):,}", "palabras distintas")
print("después->", f"{len(conteo_limpio):,}", "palabras distintas")
print()
reduccion = 100 * (1 - len(conteo_limpio) / len(conteo))
print("el vocabulario se achicó un", f"{reduccion:.0f}%")

Casi la mitad del vocabulario eran variantes de la misma palabra con distinta
puntuación o mayúscula.

Pero el ranking sigue igual de inútil.

In [ ]:
for palabra, veces in conteo_limpio.most_common(15):
    print(" ", palabra, "->", veces)

## 4. Stopwords

`datos/stopwords_es.txt` tiene una palabra por línea. **Abrilo antes de correr la
celda siguiente.** No hay nada mágico adentro: es una lista que alguien escribió.

In [ ]:
archivo = open("datos/stopwords_es.txt", encoding="utf-8")
contenido = archivo.read()
archivo.close()

palabras_vacias = contenido.split()

print("palabras vacías en la lista:", len(palabras_vacias))
print()
print("las primeras 20:")
print(palabras_vacias[:20])

In [ ]:
palabras_con_contenido = []
for palabra in palabras_limpias:
    if palabra in palabras_vacias:
        continue
    if len(palabra) < 3:
        continue
    palabras_con_contenido.append(palabra)

conteo_final = Counter(palabras_con_contenido)

print("las 20 más frecuentes:")
for palabra, veces in conteo_final.most_common(20):
    print(" ", palabra, "->", veces)

Ahora sí se puede leer algo.

> **Dos decisiones más, tomadas al pasar.** Descartamos las palabras de la lista
> **y** las de menos de tres letras. Ese `3` lo escribí yo. Con `4` el ranking
> cambia.
>
> Y una pregunta para el corpus que tenemos entre manos: ¿está `vos` en esa lista?

In [ ]:
for palabra in ["vos", "tú", "usted", "che", "yo", "él"]:
    esta = palabra in palabras_vacias
    print(" ", palabra, "-> ¿está en la lista de palabras vacías?", esta)

`tú` está y `vos` no.

La lista se armó pensando en el español peninsular. En un corpus rioplatense eso
significa que `vos` va a aparecer entre las palabras más frecuentes de tu
análisis, y `tú` no, aunque las dos sean pronombres de segunda persona.

No es un error de la lista: es que **una lista de stopwords siempre viene de algún
lado y trae los supuestos de ese lugar.**

## 5. La trampa grande: contar por año

Hasta acá contamos todo el corpus junto. Ahora la pregunta interesante: **¿cambia
lo que escribe Casciari con los años?**

Contemos la palabra `padre` año por año.

In [ ]:
def contar_palabra_por_anio(palabra_buscada):
    # Devuelve una tabla con las veces que aparece la palabra en cada año.
    filas = []
    for anio in sorted(corpus["anio"].unique()):
        del_anio = corpus[corpus["anio"] == anio]
        texto_del_anio = " ".join(del_anio["texto_limpio"])
        palabras_del_anio = texto_del_anio.split()
        fila = {
            "anio": anio,
            "veces": palabras_del_anio.count(palabra_buscada),
        }
        filas.append(fila)
    return pd.DataFrame(filas)


tabla_padre = contar_palabra_por_anio("padre")
tabla_padre

Se lee una historia: `padre` aparece 34 veces en 2007, se derrumba a 2 en 2012
y vuelve a 23 en 2015.

**Antes de creerle, una pregunta: ¿los doce años tienen el mismo tamaño?**

In [ ]:
filas = []
for anio in sorted(corpus["anio"].unique()):
    del_anio = corpus[corpus["anio"] == anio]
    texto_del_anio = " ".join(del_anio["texto_limpio"])
    fila = {
        "anio": anio,
        "palabras_del_anio": len(texto_del_anio.split()),
    }
    filas.append(fila)

tamanos = pd.DataFrame(filas)
tamanos

**No.** 2005 tiene casi cuatro veces más palabras que 2009.

Entonces el conteo crudo no compara escritura: compara **cuánto se publicó**. Un
año con el doble de texto tiene el doble de chances de contener cualquier palabra,
sin que eso signifique nada.

La solución es dividir por el tamaño. Como los números quedan chiquísimos, se
multiplica por diez mil y se dice **"apariciones por cada diez mil palabras"**.

In [ ]:
tabla = tabla_padre.merge(tamanos, on="anio")

frecuencias_relativas = []
for posicion in range(len(tabla)):
    veces = tabla.loc[posicion, "veces"]
    total_del_anio = tabla.loc[posicion, "palabras_del_anio"]
    por_diez_mil = 10000 * veces / total_del_anio
    frecuencias_relativas.append(round(por_diez_mil, 2))

tabla["por_10k"] = frecuencias_relativas
tabla

Mirá cómo cambia la historia.

Con conteos crudos, 2007 (34) parecía muy por encima de 2015 (23). Normalizados,
**9,07 contra 9,75**: 2015 está apenas arriba.

Y 2009, que con 12 apariciones parecía un año flojo, resulta ser el tercero más
alto: tiene pocas menciones porque es el año más corto del corpus.

Lo único que sobrevive intacto es 2012: **1,01 por diez mil**, un derrumbe real.

> **Esta es la operación más importante de todo el práctico.** Cabe en dos líneas
> y cambia todas las conclusiones.
>
> Cada vez que compares grupos de tamaños distintos —años, autores, secciones de
> un diario, usuarios— la pregunta es la misma: ¿estoy comparando la cosa o estoy
> comparando el tamaño de la muestra?

> ### Perilla
>
> Elegí otra palabra y volvé a correr el análisis completo. Probá con `madre`,
> `libro`, `fútbol`, `internet`, `hija`.
>
> Para cada una, mirá primero el conteo crudo y después el normalizado. **Anotá
> en qué casos la historia cambia y en cuáles no.**

In [ ]:
palabra_elegida = "internet"

tabla_elegida = contar_palabra_por_anio(palabra_elegida)
tabla_elegida = tabla_elegida.merge(tamanos, on="anio")

relativas = []
for posicion in range(len(tabla_elegida)):
    veces = tabla_elegida.loc[posicion, "veces"]
    total_del_anio = tabla_elegida.loc[posicion, "palabras_del_anio"]
    relativas.append(round(10000 * veces / total_del_anio, 2))

tabla_elegida["por_10k"] = relativas
tabla_elegida

## 6. Guardar el trabajo

El corpus limpio lo vamos a usar en los dos cuadernos que siguen. Lo guardamos
para no repetir la limpieza.

In [ ]:
corpus.to_json("datos/corpus_limpio.jsonl", orient="records", lines=True, force_ascii=False)

print("escrito: datos/corpus_limpio.jsonl")

comprobacion = pd.read_json("datos/corpus_limpio.jsonl", lines=True)
print("columnas:", list(comprobacion.columns))
print("filas:", f"{len(comprobacion):,}")

## Lo que hiciste hoy

- Contaste sin limpiar y viste que `casa`, `casa.` y `casa,` son tres palabras
  distintas para la computadora.
- Limpiaste en tres operaciones, y descartaste una cuarta que borraba 447 años.
- Aplicaste una lista de stopwords y viste que trae supuestos sobre qué español se
  habla.
- Descubriste que un conteo crudo compara tamaños de archivo, no escritura.
- Normalizaste, y la historia cambió.

## Lo que viene

En el `02` volvemos a bag of words y TF-IDF, los del teórico, pero sobre este
corpus. Y ahí vamos a medir algo incómodo sobre la herramienta.